# B3 Spectrum-RDFT + SCAL-RDFT Spectral Validation v1.0

Reviewer-requested target-blind validation on frozen `F4_to_F1`.

**Protocol SHA:** `2db80f1e58721fc1bab060df59d2232597607b600e98d367573d120f19a71172`

Critical protocol rule: the legacy `source_val` is audit-only because an actual-crop audit found 30 `source_train ↔ source_val` raw-support overlaps. The target test has zero actual-support overlap with either source subset. Training therefore uses the original 15-event `source_train` only, for a fixed 80 epochs with no early stopping or validation-selected checkpoint. Target test is opened only after epoch 80.


In [ ]:
from pathlib import Path
import os,sys,json,subprocess,shutil,base64,zipfile,platform
if not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
subprocess.run([sys.executable,'-m','pip','install','-q','pandas==2.2.3','h5py>=3.11,<4','scikit-learn>=1.5,<2'],check=True)
import pandas as pd,numpy as np,torch,h5py,sklearn
PROJECT_ROOT=Path('/content/drive/MyDrive/Project_Paper01_SCAL_DAS')
OUTPUT_ROOT=PROJECT_ROOT/'B3_Spectral_Validation_v1_0'
LOCAL_PACKAGE=Path('/content/b3_spectral_validation_v1_0')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
if not torch.cuda.is_available(): raise RuntimeError('GPU runtime required')
print('GPU:',torch.cuda.get_device_name(0))
print('Project:',PROJECT_ROOT)
print('Output:',OUTPUT_ROOT)
env={'python':sys.version,'platform':platform.platform(),'torch':torch.__version__,'cuda_runtime':torch.version.cuda,
     'gpu':torch.cuda.get_device_name(0),'pandas':pd.__version__,'numpy':np.__version__,'sklearn':sklearn.__version__,'h5py':h5py.__version__}
(OUTPUT_ROOT/'RUN_ENVIRONMENT.json').write_text(json.dumps(env,indent=2,sort_keys=True))


In [ ]:
PAYLOAD_B64='''...'''
if LOCAL_PACKAGE.exists(): shutil.rmtree(LOCAL_PACKAGE)
LOCAL_PACKAGE.mkdir(parents=True)
raw=base64.b64decode(PAYLOAD_B64.encode())
tmp=Path('/content/b3_spectral_payload.zip'); tmp.write_bytes(raw)
with zipfile.ZipFile(tmp) as z: z.extractall(LOCAL_PACKAGE)
sys.path.insert(0,str(LOCAL_PACKAGE))
from b3_spectral_campaign.config import Config,EXPECTED_PROTOCOL_SHA
from b3_spectral_campaign.campaign import bootstrap,smoke,full,aggregate,make_zip
from b3_spectral_campaign.models import RDFTPreprocessor,build_model
from b3_spectral_campaign.train import init_gate
assert EXPECTED_PROTOCOL_SHA=='2db80f1e58721fc1bab060df59d2232597607b600e98d367573d120f19a71172'
print('Embedded campaign package ready. Protocol SHA:',EXPECTED_PROTOCOL_SHA)


In [ ]:
# Preflight and data integrity
from b3_spectral_campaign.data import auto_detect_phase2
phase2=auto_detect_phase2(PROJECT_ROOT)
reg0=pd.read_csv(LOCAL_PACKAGE/'frozen'/'B3_SPECTRAL_CAMPAIGN_ASSIGNMENTS.csv')
needed=sorted(set(reg0.loc[reg0.spectral_campaign_role.isin(['train','test']),'shard_file'].astype(str)))
required=sum((phase2/r).stat().st_size for r in needed); free=shutil.disk_usage('/content').free
STAGE_LOCAL=free > required + 8*(1024**3)
print(f'Required shards: {required/1024**3:.2f} GiB; free: {free/1024**3:.2f} GiB; stage_local={STAGE_LOCAL}')
cfg,phase2_root,data_root,registry,fr=bootstrap(PROJECT_ROOT,LOCAL_PACKAGE,stage_local=STAGE_LOCAL)
print('train events/patches:',fr['train'].event_id.nunique(),len(fr['train']))
print('audit-only val events/patches:',fr['audit_val'].event_id.nunique(),len(fr['audit_val']))
print('target events/patches:',fr['test'].event_id.nunique(),len(fr['test']))
print('TARGET ISOLATION GATE: PASS')


In [ ]:
# Model/RDFT/initialization gates
for seed in cfg.seeds: print(init_gate(seed,cfg))
pre=RDFTPreprocessor().cuda().eval()
x=torch.zeros(1,1,8192,128,device='cuda')
with torch.no_grad(): sx=pre(x)
assert sx.shape==(1,1,2048,128) and torch.isfinite(sx).all()
fs=20000.; t=torch.arange(8192,device='cuda')/fs
sig=torch.sin(2*torch.pi*100*t); xx=sig[None,None,:,None].repeat(1,1,1,128)
with torch.no_grad(): s=pre(xx)[0,0,:,0]
expected=round(100/(fs/(8192*3))); observed=int(torch.argmax(s))
print('100-Hz bin expected/observed:',expected,observed); assert abs(expected-observed)<=1
del pre,x,sx,xx,s; torch.cuda.empty_cache()
print('MODEL / RDFT / INITIALIZATION GATES: PASS')


In [ ]:
# Two-epoch seed-2026 smoke: source train only, NO target loader.
gate=smoke(fr,data_root,cfg,OUTPUT_ROOT)
print(json.dumps(gate,indent=2))
assert gate['status']=='PASS' and gate['test_accessed'] is False
print('B3 SPECTRAL SMOKE GATE: PASS')


## Full campaign

The next cell runs **2 spectral models × 3 seeds × 80 fixed epochs**. It is resume-safe. There is no early stopping and no source-validation checkpoint selection. Target test is evaluated only after each 80-epoch source-only trajectory finishes.


In [ ]:
runs=full(fr,data_root,cfg,OUTPUT_ROOT)
print('Completed:',len(runs))
per,agg=aggregate(OUTPUT_ROOT,cfg)
display(per.sort_values(['model','seed']))
display(agg)
out_zip,out_sha=make_zip(OUTPUT_ROOT,LOCAL_PACKAGE)
print('\nB3 SPECTRAL VALIDATION COMPLETE')
print('Audit ZIP:',out_zip)
print('SHA256:',out_sha)
print('Upload B3_SPECTRAL_VALIDATION_AUDIT.zip back to ChatGPT.')
